# Bottle Vision — SAM 3 Colab Runner

This notebook uses **one normal Colab Python runtime**. It does not create a second Python/Conda/uv environment, because keeping Colab's CUDA runtime and PyTorch in one environment is the least fragile path.

Start with **Runtime → Change runtime type → GPU** and preferably a fresh runtime.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/content/Bottlevision')
REMOTE = 'https://github.com/Rollerboy22/Bottlevision.git'
if REPO.exists() and (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
elif REPO.exists():
    raise RuntimeError(f'{REPO} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REMOTE, str(REPO)], check=True)
%cd /content/Bottlevision
print('Repository:', REPO)
print('Python:', sys.version)

## 1. Install the stack without replacing Colab's Python

SAM 3 currently documents Python 3.12+, PyTorch 2.7+, and CUDA 12.6+. The current upstream package also declares `numpy<2`; `einops` is required by the SAM transformer code and is installed explicitly here.

We use `--no-deps` for the editable project installs so pip cannot silently replace the CUDA PyTorch stack while resolving unrelated dependencies.

In [ ]:
import sys
import subprocess

if sys.version_info < (3, 12):
    raise RuntimeError(f'SAM 3 requires Python 3.12+. Colab is using {sys.version}')

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

pip('install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel')
pip('install', '-q', '--upgrade', 'numpy<2')
pip('install', '-q', '--upgrade', 'torch==2.10.0', 'torchvision==0.25.0', '--index-url', 'https://download.pytorch.org/whl/cu128')
pip('install', '-q', 'pydantic>=2.7,<3', 'PyYAML>=6,<7', 'Pillow>=10,<12')
pip('install', '-q', 'timm>=1.0.17', 'tqdm', 'ftfy==6.1.1', 'regex', 'iopath>=0.1.10', 'typing_extensions', 'huggingface_hub>=0.30', 'einops>=0.8')
pip('install', '-q', '-e', '.', '--no-deps')
pip('install', '-q', '-e', 'git+https://github.com/facebookresearch/sam3.git#egg=sam3', '--no-deps')
print('Installation finished.')

In [ ]:
import sys
import numpy as np
import torch
import torchvision
import sam3

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('PyTorch:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('TorchVision:', torchvision.__version__)
print('SAM3:', sam3.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a Colab GPU runtime and restart the session.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU capability:', torch.cuda.get_device_capability(0))
from einops import rearrange
print('einops: OK')
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
print('SAM3 API imports: OK')

## 2. Optional Hugging Face authentication

SAM 3 checkpoints require access to the SAM 3 Hugging Face repository. If your account has already been authenticated in this runtime, skip this cell. Otherwise uncomment and run it, then paste your Hugging Face token when prompted.

In [ ]:
# from huggingface_hub import login
# login()

In [ ]:
import importlib.util
import sys
from pathlib import Path

RUNNER_PATH = REPO / 'colab' / 'BottleVision_SAM3_Runner.py'
sys.path.insert(0, str(REPO))
spec = importlib.util.spec_from_file_location('bottle_vision_colab_runner', RUNNER_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not load runner module from {RUNNER_PATH}')
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
load_bottle_vision_config = runner.load_bottle_vision_config
load_rgb_image = runner.load_rgb_image
run_segmentation = runner.run_segmentation
build_review_views = runner.build_review_views
summarize_result = runner.summarize_result
config = load_bottle_vision_config(REPO / 'configs' / 'default.yaml')
print('Bottle Vision config loaded.')

In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No image was uploaded.')
IMAGE_PATH = next(iter(uploaded))
image = load_rgb_image(IMAGE_PATH)
print('Image:', IMAGE_PATH, image.shape)
result = run_segmentation(image, config)
summary = summarize_result(result)
summary

In [ ]:
import matplotlib.pyplot as plt
views = build_review_views(image, result, alpha=0.45)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for axis, (title, view) in zip(axes, views.items()):
    axis.imshow(view)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

## Human review checkpoint

Never promote a bad automatic mask. The canonical annotation is the pixel mask, not a bounding box. Low-confidence or low-quality instances remain candidates for review/resegmentation.